# Assignments - Numerical greeks
5. assignment notebook

Suppose $S_{1}(t)$ and $S_{2}(t)$ are independent stock processes with equal volatility. $S_{1}(0)=S_{2}(0)=100$.
The contract at maturity pays the value of the most expensive stock: $V(T)=max\{S_{1}(T),S_{2}(T)\}$.

* What is the formula for the fair value of the contract, $V(0)$?
* Derive the partial differential equation for this contract
* Solve the resultant PDE using Finite Difference Method
* Compare the results obtained through solving the PDE with Monte Carlo Method

$V(0)=S_{1}(0)\Phi(d_{1})+S_{2}(0)\Phi(d_{2})$

$d_{1}=\frac{ln(S_{1}/S_{2})+\frac{1}{2}\hat{\sigma}^{2}T}{\hat{\sigma}\sqrt{T}}$

$d_{2}=\frac{ln(S_{2}/S_{1})+\frac{1}{2}\hat{\sigma}^{2}T}{\hat{\sigma}\sqrt{T}}$

$\hat{\sigma}=\sigma\sqrt{2}$

a standard normális eloszlásfüggvény.

$V(0)=100\cdot\Phi(\frac{\sigma\sqrt{T}}{\sqrt{2}})+100\cdot\Phi(\frac{\sigma\sqrt{T}}{\sqrt{2}})=200\cdot\Phi(\frac{\sigma\sqrt{T}}{\sqrt{2}})$

## 1. The partial differential equation for this contract

Since $V=V(S_{1},S_{2},t)$

$\frac{\partial V}{\partial t}+rS_{1}\frac{\partial V}{\partial S_{1}}+rS_{2}\frac{\partial V}{\partial S_{2}}+\frac{1}{2}\sigma^{2}S_{1}^{2}\frac{\partial^{2}V}{\partial S_{1}^{2}}+\frac{1}{2}\sigma^{2}S_{2}^{2}\frac{\partial^{2}V}{\partial S_{2}^{2}}-rV=0$

By substituting $x=log(S_{1}/K)$ and $y=log(S_{2}/K)$, $V=V(x,y,t)$.

$V_{t}+(r-\frac{\sigma^{2}}{2})V_{x}+(r-\frac{\sigma^{2}}{2})V_{y}+\frac{\sigma^{2}}{2}V_{xx}+\frac{\sigma^{2}}{2}V_{yy}-rV=0$

The boundary conditions:
$V(T,x,y)=K~max(e^{x},e^{y})$.

$V(t,x_{min},y)\approx Ke^{y}$, $V(t,x_{max},y)\approx Ke^{x_{max}}$,
$V(t,x,y_{min})\approx Ke^{x}$ , $V(t,x,y_{max})\approx Ke^{y_{max}}$

## 2. Analytical solution

$d_{1}=\frac{ln(S_{1}/S_{2})+\frac{1}{2}\hat{\sigma}^{2}T}{\hat{\sigma}\sqrt{T}}$

$d_{2}=\frac{ln(S_{2}/S_{1})+\frac{1}{2}\hat{\sigma}^{2}T}{\hat{\sigma}\sqrt{T}}$

$V(0)=S_{1}(0)\Phi(d_{1})+S_{2}(0)\Phi(d_{2})$

$\hat{\sigma}=\sigma\sqrt{2}$

standard normal distribution.

$V(0)=100\cdot\Phi(\frac{\sigma\sqrt{T}}{\sqrt{2}})+100\cdot\Phi(\frac{\sigma\sqrt{T}}{\sqrt{2}})=200\cdot\Phi(\frac{\sigma\sqrt{T}}{\sqrt{2}})$

In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from tqdm import tqdm

def analytic_max_equal_spots(S0, T, sigma):
    # exact formula for equal spots
    return 2.0 * S0 * norm.cdf(sigma * np.sqrt(T) / np.sqrt(2.0))

price_an = analytic_max_equal_spots(100.0, 1.0, 0.20)
print(price_an)

111.24629160182849


## 3. Numerical scheme

In [2]:
# setting up the params for the numerical grid
K = 100.0
T = 1.0
R = 0.05
SIGMA = 0.20

x_min = -4
x_max = 4
y_min = -4
y_max = 4
n_x = 201
n_y = 201
n_t = 201

In [3]:
def explicit_stability_report(x_min, x_max, y_min, y_max, T, sigma, n_x, n_y, n_t, safety_limit=0.5):
    # simple check to see if the explicit scheme will blow up
    dx = (x_max - x_min) / (n_x - 1)
    dy = (y_max - y_min) / (n_y - 1)
    dt = T / (n_t - 1)
    
    coeff = 0.5 * sigma**2 * (1.0 / dx**2 + 1.0 / dy**2)
    theta = coeff * dt
    dt_max = safety_limit / coeff
    n_t_min = math.ceil(T / dt_max) + 1
    
    print(f"dx = {dx:.6f}")
    print(f"dy = {dy:.6f}")
    print(f"dt = {dt:.6f}")
    print(f"theta = {theta:.6f} (<= 0.5?)")
    print(f"max allowed dt for safety_limit={safety_limit}: {dt_max:.6f}")
    print(f"recommended minimum n_t: {n_t_min}")
    
    if theta <= safety_limit:
        print("Stability check: OK")
    else:
        print("WARNING: likely unstable")
        
    return {
        "dx": dx,
        "dy": dy,
        "dt": dt,
        "theta": theta,
        "dt_max": dt_max,
        "n_t_min": n_t_min,
    }

report = explicit_stability_report(
    x_min, x_max, y_min, y_max, T, SIGMA, n_x, n_y, n_t
)

dx = 0.040000
dy = 0.040000
dt = 0.005000
theta = 0.125000 (<= 0.5?)
max allowed dt for safety_limit=0.5: 0.020000
recommended minimum n_t: 52
Stability check: OK


In [4]:
def boundary_max(grid, x_min, x_max, y_min, y_max, K, T):
    # filling out the edges of the grid based on the boundary conditions
    n_x, n_y, n_t = grid.shape
    x_values = np.linspace(x_min, x_max, n_x)
    y_values = np.linspace(y_min, y_max, n_y)
    
    # terminal condition at t=T
    for i, x in enumerate(x_values):
        for j, y in enumerate(y_values):
            grid[i, j, -1] = K * max(np.exp(x), np.exp(y))
            
    # boundaries for all times
    for n in range(n_t):
        # x = x_min and x = x_max
        for j, y in enumerate(y_values):
            grid[0, j, n] = K * np.exp(y)
            grid[-1, j, n] = K * np.exp(x_max)
            
        # y = y_min and y = y_max
        for i, x in enumerate(x_values):
            grid[i, 0, n] = K * np.exp(x)
            grid[i, -1, n] = K * np.exp(y_max)
            
    return grid

In [5]:
def get_max_price_t0(grid, S1_0, S2_0, K):
    # grabbing the specific price from the grid at t=0
    x_values = np.linspace(x_min, x_max, grid.shape[0])
    y_values = np.linspace(y_min, y_max, grid.shape[1])
    
    x0 = np.log(S1_0 / K)
    y0 = np.log(S2_0 / K)
    
    ix0 = np.argmin(np.abs(x_values - x0))
    iy0 = np.argmin(np.abs(y_values - y0))
    
    return grid[ix0, iy0, 0]

## 4. Vectorized numerical scheme
(I am just skipping the double-for-loop version because it takes way too long to run)

In [6]:
def create_and_run_explicit_max_vectorized(n_x, n_y, n_t):
    dx = (x_max - x_min) / (n_x - 1)
    dy = (y_max - y_min) / (n_y - 1)
    dt = T / (n_t - 1)
    
    grid = np.zeros((n_x, n_y, n_t))
    grid = boundary_max(grid, x_min, x_max, y_min, y_max, K, T)
    
    a = R - 0.5 * SIGMA**2
    b = 0.5 * SIGMA**2
    
    # stepping backwards in time
    for n in tqdm(range(n_t - 1, 0, -1), desc="Time stepping"):
        g = grid[:, :, n]
        
        # central differences for first derivs
        vx = (g[2:, 1:-1] - g[:-2, 1:-1]) / (2.0 * dx)
        vy = (g[1:-1, 2:] - g[1:-1, :-2]) / (2.0 * dy)
        
        # central differences for second derivs
        vxx = (g[2:, 1:-1] - 2.0 * g[1:-1, 1:-1] + g[:-2, 1:-1]) / dx**2
        vyy = (g[1:-1, 2:] - 2.0 * g[1:-1, 1:-1] + g[1:-1, :-2]) / dy**2
        
        # explicit euler step
        grid[1:-1, 1:-1, n - 1] = g[1:-1, 1:-1] + dt * (
            a * vx + a * vy + b * vxx + b * vyy - R * g[1:-1, 1:-1]
        )
        
    return grid

In [7]:
grid = create_and_run_explicit_max_vectorized(n_x=n_x, n_y=n_y, n_t=n_t)
price_num = get_max_price_t0(grid, S1_0=100.0, S2_0=100.0, K=K)
price_an = analytic_max_equal_spots(100.0, 1.0, 0.20)

print("Numerical PDE price:", price_num)
print("Analytic price     :", price_an)
print("Absolute error     :", abs(price_num - price_an))

Time stepping: 100%|██████████| 200/200 [00:00<00:00, 515.73it/s]

Numerical PDE price: 111.23129001141662
Analytic price     : 111.24629160182849
Absolute error     : 0.015001590411870325


## 5. Monte Carlo

In [8]:
def monte_carlo_bestof_two_assets(S1_0, S2_0, T, r, sigma, n_sim, seed=None):
    if seed is not None:
        np.random.seed(seed)
        
    Z1 = np.random.randn(n_sim)
    Z2 = np.random.randn(n_sim)
    
    # simulate the two independent stocks
    S1_T = S1_0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z1)
    S2_T = S2_0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z2)
    
    payoffs = np.maximum(S1_T, S2_T)
    discounted_payoffs = np.exp(-r * T) * payoffs
    
    price_estimate = np.mean(discounted_payoffs)
    std_error = np.std(discounted_payoffs, ddof=1) / np.sqrt(n_sim)
    
    conf_low = price_estimate - 1.96 * std_error
    conf_high = price_estimate + 1.96 * std_error
    
    return price_estimate, std_error, conf_low, conf_high

In [9]:
price_mc, se_mc, ci_low, ci_high = monte_carlo_bestof_two_assets(
    S1_0=100.0,
    S2_0=100.0,
    T=T,
    r=R,
    sigma=SIGMA,
    n_sim=200000,
    seed=42
)

print("Monte Carlo price:", price_mc)
print("Std. error       :", se_mc)
print("95% CI           :", (ci_low, ci_high))
print("Analytic price   :", price_an)
print("Abs error        :", abs(price_mc - price_an))

Monte Carlo price: 111.25896185839767
Std. error       : 0.04179121376775127
95% CI           : (np.float64(111.17705107941288), np.float64(111.34087263738246))
Analytic price   : 111.24629160182849
Abs error        : 0.012670256569180083


# Conclusion

Both the vectorized explicit Finite Difference Method (111.23) and the Monte Carlo simulation (111.26) successfully approximate the true analytical price (111.25). The vectorized PDE solves almost instantly but requires a complex grid setup and is bound by strict stability limits. In contrast, the Monte Carlo method is simpler to implement and avoids grid boundary issues, but it requires a massive number of random simulations (200,000) to shrink the standard error and bring the 95% confidence interval into an acceptable range.